# External DICOM-to-NIfTI conversion

**Scientific purpose.** Inventory private HCC-TACE-Seg DICOM series, identify CT phase
candidates, and convert CT series to NIfTI while preserving their source geometry.

**Runnability classification:** runnable after notebook 07 with private DICOM access.
**Inputs:** privately acquired HCC-TACE-Seg DICOM series.
**Private outputs:** DICOM catalog, CT conversion manifest, and NIfTI CT volumes.

No registration occurs here. Phase-description heuristics are recorded for review, and
uncertain series remain `unknown` instead of being silently assigned.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import re
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

EXTERNAL_ROOT = require_path(CONFIG, "hcc_tace_seg_root")
RAW_DICOM_ROOT = EXTERNAL_ROOT / "raw_dicom"
CONVERSION_ROOT = PRIVATE_ARTIFACT_ROOT / "external" / "ct_conversion"
CT_NIFTI_ROOT = CONVERSION_ROOT / "ct_nifti"
CATALOG_PATH = CONVERSION_ROOT / "dicom_catalog.csv"
CONVERSION_PATH = CONVERSION_ROOT / "ct_conversion.csv"
if not RAW_DICOM_ROOT.is_dir():
    raise FileNotFoundError("Private HCC-TACE-Seg DICOM root is unavailable; run notebook 07.")
CT_NIFTI_ROOT.mkdir(parents=True, exist_ok=True)


## Private DICOM catalog

UIDs, patient keys, and source paths are stored only in the configured private artifact
root. Notebook output is limited to aggregate modality counts.


In [ ]:
def header_value(dataset, name, default=""):
    value = getattr(dataset, name, default)
    return str(value) if value is not None else str(default)


def build_dicom_catalog(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        try:
            header = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        except Exception:
            continue
        if not hasattr(header, "SeriesInstanceUID"):
            continue
        position = getattr(header, "ImagePositionPatient", None)
        rows.append(
            {
                "patient_key": header_value(header, "PatientID"),
                "series_uid": header_value(header, "SeriesInstanceUID"),
                "sop_uid": header_value(header, "SOPInstanceUID"),
                "modality": header_value(header, "Modality").upper(),
                "series_description": header_value(header, "SeriesDescription"),
                "protocol_name": header_value(header, "ProtocolName"),
                "image_position": json.dumps([float(value) for value in position])
                if position is not None
                else "",
                "file_path": str(path),
            }
        )
    catalog = pd.DataFrame(rows)
    if catalog.empty:
        raise RuntimeError("No readable DICOM series were found in private storage.")
    catalog.to_csv(CATALOG_PATH, index=False)
    return catalog


catalog = build_dicom_catalog(RAW_DICOM_ROOT)
catalog.groupby("modality").size().rename("instances").reset_index()


## CT phase assignment and conversion

The phase labels are P (noncontrast), C1 (arterial), C2 (portal venous), C3 (delayed),
or unknown. The external source is heterogeneous; no patient in the retained cohort had
all four required phases.


In [ ]:
def normalized_description(row: pd.Series) -> str:
    return re.sub(
        r"\s+",
        " ",
        f"{row.get('series_description', '')} {row.get('protocol_name', '')}".lower(),
    ).strip()


def infer_phase(row: pd.Series) -> str:
    description = normalized_description(row)
    if any(token in description for token in ("non contrast", "noncontrast", "pre contrast", "plain")):
        return "P"
    if any(token in description for token in ("arterial", "art phase")):
        return "C1"
    if any(token in description for token in ("portal", "venous", "pv phase")):
        return "C2"
    if any(token in description for token in ("delay", "delayed", "equilibrium")):
        return "C3"
    return "unknown"


def convert_ct_series(series_uid: str, patient_key: str, phase: str, files) -> dict:
    ordered_files = list(
        sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(Path(files[0]).parent), series_uid)
    )
    if not ordered_files:
        ordered_files = sorted(map(str, files))
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(ordered_files)
    image = reader.Execute()
    output_path = CT_NIFTI_ROOT / f"{patient_key}__{series_uid}.nii.gz"
    sitk.WriteImage(image, str(output_path), useCompression=True)
    ordered_sop_uids = []
    ordered_positions = []
    for path in ordered_files:
        header = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        ordered_sop_uids.append(header_value(header, "SOPInstanceUID"))
        position = getattr(header, "ImagePositionPatient", None)
        ordered_positions.append(
            [float(value) for value in position] if position is not None else None
        )
    return {
        "patient_key": patient_key,
        "series_uid": series_uid,
        "phase": phase,
        "ct_path": str(output_path),
        "shape_xyz": json.dumps(list(image.GetSize())),
        "spacing_xyz": json.dumps(list(image.GetSpacing())),
        "ordered_sop_uids": json.dumps(ordered_sop_uids),
        "ordered_positions": json.dumps(ordered_positions),
    }


def convert_all_ct_series(catalog_table: pd.DataFrame) -> pd.DataFrame:
    records = []
    ct_catalog = catalog_table[catalog_table["modality"] == "CT"]
    for (patient_key, series_uid), group in ct_catalog.groupby(["patient_key", "series_uid"]):
        representative = group.iloc[0]
        records.append(
            convert_ct_series(
                str(series_uid),
                str(patient_key),
                infer_phase(representative),
                group["file_path"].tolist(),
            )
        )
    converted = pd.DataFrame(records)
    converted.to_csv(CONVERSION_PATH, index=False)
    return converted


# Conversion is intentionally explicit because it can require substantial private storage.
# Run the following function after reviewing aggregate modality counts:
# converted_ct = convert_all_ct_series(catalog)


## Review requirement

Review unknown or duplicated phase candidates in private storage before downstream mask
mapping. Do not publish the catalog or conversion manifest: both contain patient-linked
DICOM metadata. Conversion preserves DICOM geometry but does not make the external
adapter equivalent to the historical internal preprocessing.
